In [1]:
import os
import io
import spacy
!pip install docx2txt
!pip install pdfminer.six
!pip install nltk
!pip install pandas
import docx2txt
from pdfminer.layout import LAParams
from pdfminer.pdfpage import PDFPage
from pdfminer.pdfparser import PDFSyntaxError
from nltk.corpus import stopwords
import multiprocessing as mp
import pprint
from spacy.matcher import Matcher
import re
import nltk
import pandas as pd
from datetime import datetime
from dateutil import relativedelta
from pdfminer.converter import TextConverter
from pdfminer.pdfinterp import PDFPageInterpreter
from pdfminer.pdfinterp import PDFResourceManager
from nltk.stem import WordNetLemmatizer
from pdfminer.high_level import extract_text
import numpy as np

  Using cached docx2txt-0.8-py3-none-any.whl
  Using cached pdfminer.six-20201018-py3-none-any.whl (5.6 MB)
  Using cached chardet-4.0.0-py2.py3-none-any.whl (178 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Using cached cryptography-3.4.8-cp36-abi3-win_amd64.whl (1.6 MB)
  Using cached nltk-3.6.2-py3-none-any.whl (1.5 MB)
  Using cached joblib-1.0.1-py3-none-any.whl (303 kB)
  Using cached regex-2021.8.21-cp37-cp37m-win_amd64.whl (270 kB)
  Using cached pandas-1.3.2-cp37-cp37m-win_amd64.whl (10.0 MB)
  Using cached pytz-2021.1-py2.py3-none-any.whl (510 kB)


In [2]:
def extension(resume):
    if not isinstance(resume, io.BytesIO):
        ext = os.path.splitext(resume)[1].split('.')[1]
    else:
        ext = resume.name.split('.')[1]
    return ext

def extract_text_from_pdf(file_path):
    return extract_text(file_path)

def extract_text_from_doc(file_path):
    temp = docx2txt.process(file_path)
    text = [line.replace('\t', ' ') for line in temp.split('\n') if line]
    return '\n'.join(text)

def extraction(resume):
    ext = extension(resume)
    text = ""
    if ext == "docx":
        text = extract_text_from_doc(resume)
    if ext == "pdf":
        text = extract_text_from_pdf(resume)
    return text

NAME_PATTERN = [{'POS': 'PROPN'}, {'POS': 'PROPN'}]
NOT_ALPHA_NUMERIC = r'[^a-zA-Z\d]'
NUMBER = r'\d+'

nlp = spacy.load('en_core_web_sm')
matcher = Matcher(nlp.vocab)

def extract_name(text, matcher):
    nlp_text = nlp(text)
    pattern = NAME_PATTERN
    matcher.add('NAME', [pattern])
    matches = matcher(nlp_text)
    for _, start, end in matches:
        span = nlp_text[start:end]
        if 'name' not in span.text.lower():
            return span.text

def extract_email(text):
    email = re.findall(r"([^@|\s]+@[^@]+\.[^@|\s]+)", text)
    if email:
        try:
            return email[0].split()[0].strip(';')
        except IndexError:
            return None

def extract_mobile_number(text, custom_regex=None):
    mob_num_regex = r'''(\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4}|\(\d{3}\)
                        [-\.\s]*\d{3}[-\.\s]??\d{4}|\d{3}[-\.\s]??\d{4})'''
    phone = re.findall(re.compile(mob_num_regex), text)
    if phone:
        number = ''.join(phone[0])
        return number
    
EDUCATION = ['BE', 'B.E.', 'B.E', 'BS','B.SC', 'B.S', 'M.E','M.E.', 'MS', 'M.S', 'M.SC', 'BTECH', 'MTECH', 'SSC', 'HSC', 'CBSE', 'ICSE', 'XII', "BACHELOR OF TECHNOLOGY", "BACHELOR OF SCIENCE", "BA", "B.A", "MASTER OF TECHNOLOGY", "MASTER OF SCIENCE"]
STOPWORDS = set(stopwords.words('english'))
YEAR = r'(((20|19)(\d{2})))'
SUBJECT = ["I.T", "IT", "CSE", "MECHANICAL ENGINEERING","COMPUTER ENGINEERING", "ELECTRICAL ENGINEERING", "CSE", "EECS", "COMPUTER SCIENCE", "MATHEMATICS", "STATISTICS"]

def extract_education(text):
    word_tokens = nltk.tokenize.word_tokenize(text)
    bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))

    # we create a set to keep the results in.
    found_degrees = []

    # we search for each token in our skills database
    for token in  word_tokens:
        if token.upper() in EDUCATION:
            found_degrees.append(token.upper())

    # we search for each bigram and trigram in our skills database
    for ngram in bigrams_trigrams:
        if ngram.upper() in EDUCATION:
            found_degrees.append(ngram.upper())

    return found_degrees

def extract_subject(text):
    word_tokens = nltk.tokenize.word_tokenize(text)
    found_subjects = []

    # we search for each token in our skills database
    for token in  word_tokens:
        if token.upper() in SUBJECT:
            found_subjects.append(token.upper())

    return found_subjects

DESIGNATION = ["DATA SCIENTIST","DATA ENGINEER","SOFTWARE ENGINEER","DATA ANALYST","MACHINE LEARNING ENGINEER","ANALYST","ASSOCIATE","SR. ASSOCIATE","SENIOR ASSOCIATE","VICE-PRESIDENT","SVP","VP",
               "SENIOR VICE PRESIDENT", "DIRECTOR", "ASSISTANT DIRECTOR", "EXECUTIVE DIRECTOR", "INTERN", "PARTNER", "HEAD"]

def extract_designation(text):
    word_tokens = nltk.tokenize.word_tokenize(text)
    bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))

    found_designations = []

    for token in  word_tokens:
        if token.upper() in DESIGNATION:
            found_designations.append(token)

    return found_designations

time_list = ["PRESENT"]
    
def extract_time(text):
    spacy_nlp  = spacy.load('en_core_web_sm')
    doc = spacy_nlp(text.strip())
    word_tokens = nltk.tokenize.word_tokenize(text)
    bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))
    
    time_indicator_entities = []
        
    for token in  word_tokens:
        if token.upper() in time_list:
            time_indicator_entities.append(token)
    
    for i in doc.ents:
        entry = str(i.lemma_).lower()
        text = text.replace(str(i).lower(), "")
                        
        if i.label_ in ["TIM", "DATE"]:
            time_indicator_entities.append(entry)

    return time_indicator_entities

def experience(dates):
    try:
        pairs = tuple((i, i+1) for i in range(len(dates)))
        pair_index = []
        for i in range(len(pairs)):
            if i%2 == 0:
                pair_index.append(i)
                
        new_pairs = []
        for i in pair_index:
            new_pairs.append(pairs[i])
            
        monthsofexperience = []
            
        for i,j in new_pairs:
            monthsofexperience.append(get_number_of_months_from_dates(dates[i], dates[j]))
            
        monthsofexperience = np.absolute(monthsofexperience)
        experience_years = sum(monthsofexperience)/12
        
        return abs(experience_years)
    except:
        return 0
    
def get_number_of_months_from_dates(date1, date2):
    if date2.lower() == 'present':
        date2 = datetime.now().strftime('%b %Y')
    try:
        if len(date1.split()[0]) > 3:
            date1 = date1.split()
            date1 = date1[0][:3] + ' ' + date1[1]
        if len(date2.split()[0]) > 3:
            date2 = date2.split()
            date2 = date2[0][:3] + ' ' + date2[1]
    except IndexError:
        return 0
    try:
        date1 = datetime.strptime(str(date1), '%b %Y')
        date2 = datetime.strptime(str(date2), '%b %Y')
        months_of_experience = relativedelta.relativedelta(date2, date1)
        months_of_experience = (months_of_experience.years * 12 + months_of_experience.months)
    except ValueError:
        return 0
    return months_of_experience

SKILLS_DB = ['machine learning','data science','python', 'word','excel', 'deep learning', "data analytics", "time series", "financial analysis", "statistical modelling", 'python', 'r', 'mysql', 'powerbi', 'flask', 'html', 'azure ml Studio', 'microsoft office', 'visual studio', 'pyspark','tensorflow', "java", "c++", "javascript", "c"]

def extract_skills(text):
    word_tokens = nltk.tokenize.word_tokenize(text)
    bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))

    # we create a set to keep the results in.
    found_skills = []

    # we search for each token in our skills database
    for token in  word_tokens:
        if token.lower() in SKILLS_DB:
            found_skills.append(token.lower())

    # we search for each bigram and trigram in our skills database
    for ngram in bigrams_trigrams:
        if ngram.lower() in SKILLS_DB:
            found_skills.append(ngram.lower())

    return found_skills

In [3]:
class ResumeParserTool(object):
    def __init__(self, file_path, custom_regex=None):
        nlp = spacy.load("en_core_web_sm")
        self.__matcher = Matcher(nlp.vocab) 
        self.__details = {'name': None,'email': None,'mobile_number': None,'skills': None,'degree': None,'designation': None,'total_experience': None, 'area_of_study': None}
        self.__file_path = file_path
        self.__text = extraction(self.__file_path)
        self.__nlp = nlp(self.__text)
        self.__noun_chunks = list(self.__nlp.noun_chunks)
        self.__get_basic_details()
        
    def get_extracted_data(self):
        return self.__details
    
    def __get_basic_details(self):
        name = extract_name(self.__text, matcher=self.__matcher)
        email = extract_email(self.__text)
        mobile = extract_mobile_number(self.__text, custom_regex = None)
        skills = extract_skills(self.__text)
        dates = extract_time(self.__text)
        for i in dates:
            special = ["annually", "monthly", "daily", "yearly", "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009","2010","2011","2012","2013","2014","2015","2016","2017","2018","2020","2021","2022","2019"]
            if i in special:
                dates.remove(i)
        total_experience = abs(experience(dates))
        designations = extract_designation(self.__text)
        degrees = extract_education(self.__text)
        subjects = extract_subject(self.__text)
        
        #extract names
        self.__details['name'] = name
        # extract email
        self.__details['email'] = email
        # extract mobile number
        self.__details['mobile_number'] = mobile
        # extract skills
        self.__details['skills'] = skills
        # extract degrees 
        self.__details["degree"] = degrees
        # extract designations
        self.__details["designation"] = designations
        # extract total experience 
        self.__details['total_experience'] = total_experience 
        # extract subjects
        self.__details["area_of_study"] = subjects
        
    def extension(file_path):
        if not isinstance(self.__file_path, io.BytesIO):
            ext = os.path.splitext(self.__file_path)[1].split('.')[1]
            
        else:
            ext = self.__file_path.name.split('.')[1]
            
        return ext
    
    def extract_text_from_pdf(file_path):
        return extract_text(self.__file_path)
    
    def extract_text_from_doc(file_path):
        temp = docx2txt.process(self.__file_path)
        text = [line.replace('\t', ' ') for line in temp.split('\n') if line]
        return '\n'.join(text)
    
    def extraction(file_path):
        ext = extension(self.__file_path)
        text = ""
        if ext == "docx":
            text = extract_text_from_doc(self.__file_path)
        if ext == "pdf":
            text = extract_text_from_pdf(self.__file_path)
        return text
    
    NAME_PATTERN = [{'POS': 'PROPN'}, {'POS': 'PROPN'}]
    NOT_ALPHA_NUMERIC = r'[^a-zA-Z\d]'
    NUMBER = r'\d+'
    nlp = spacy.load('en_core_web_sm')
    matcher = Matcher(nlp.vocab)
    
    def extract_name(self, text, matcher):
        nlp_text = nlp(self.__text)
        pattern = NAME_PATTERN
        matcher.add('NAME', [pattern])
        matches = matcher(nlp_text)
        for _, start, end in matches:
            span = nlp_text[start:end]
            if 'name' not in span.text.lower():
                return span.text
            
    def extract_email(text):
        email = re.findall(r"([^@|\s]+@[^@]+\.[^@|\s]+)", self.__text)
        if email:
            try:
                return email[0].split()[0].strip(';')
            except IndexError:
                return None
            
    def extract_mobile_number(text, custom_regex=None):
        mob_num_regex = r'''(\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4}|\(\d{3}\)
                        [-\.\s]*\d{3}[-\.\s]??\d{4}|\d{3}[-\.\s]??\d{4})'''
        phone = re.findall(re.compile(mob_num_regex), self.__text)
        if phone:
            number = ''.join(phone[0])
            return number
        
    EDUCATION = ['BE', 'B.E.', 'B.E', 'BS','B.SC',"B.SC.", 'B.S', 'M.E','M.E.', 'MS', 'M.S', 'M.SC', 'BTECH', 'MTECH', 'SSC', 'HSC', 'CBSE', 'ICSE', 'XII', "BACHELOR OF TECHNOLOGY", "BACHELOR OF SCIENCE", "BA", "B.A", "MASTER OF TECHNOLOGY", "MASTER OF SCIENCE"]
    STOPWORDS = set(stopwords.words('english'))
    YEAR = r'(((20|19)(\d{2})))'
    SUBJECT = ["I.T", "IT", "CSE", "MECHANICAL ENGINEERING", "ELECTRICAL ENGINEERING", "CSE", "EECS", "COMPUTER SCIENCE", "MATHEMATICS", "STATISTICS"]

    def extract_education(text):
        word_tokens = nltk.tokenize.word_tokenize(text)
        found_degrees = []
        for token in  word_tokens:
            if token.upper() in EDUCATION:
                found_degrees.append(token.upper())
        return found_degrees

    def extract_subject(text):
        word_tokens = nltk.tokenize.word_tokenize(text)
        found_subjects = []
        for token in  word_tokens:
            if token.upper() in SUBJECT:
                found_subjects.append(token.upper())
        return found_subjects
    
    DESIGNATION = ["DATA SCIENTIST","DATA ENGINEER","SOFTWARE ENGINEER","DATA ANALYST","MACHINE LEARNING ENGINEER","ANALYST","ASSOCIATE","SR. ASSOCIATE","SENIOR ASSOCIATE","VICE-PRESIDENT","SVP","VP",
               "SENIOR VICE PRESIDENT", "DIRECTOR", "ASSISTANT DIRECTOR", "EXECUTIVE DIRECTOR", "INTERN", "PARTNER", "HEAD"]

    def extract_designation(text):
        word_tokens = nltk.tokenize.word_tokenize(text)
        bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))

        found_designations = []

        for token in  word_tokens:
            if token.upper() in DESIGNATION:
                found_designations.append(token)

        return found_designations
    
    time_list = ["PRESENT"]
    
    def extract_time(text):
        spacy_nlp  = spacy.load('en_core_web_sm')
        doc = spacy_nlp(text.strip())
        word_tokens = nltk.tokenize.word_tokenize(text)
        bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))
    
        time_indicator_entities = []
        
        for token in  word_tokens:
            if token.upper() in time_list:
                time_indicator_entities.append(token.lower())
    
        for i in doc.ents:
            entry = str(i.lemma_).lower()
            text = text.replace(str(i).lower(), "")
                        
            if i.label_ in ["TIM", "DATE"]:
                time_indicator_entities.append(entry)

        return time_indicator_entities
        
    def experience(dates):
        try:
            pairs = tuple((i, i+1) for i in range(len(dates)))
            pair_index = []
            for i in range(len(pairs)):
                if i%2 == 0:
                    pair_index.append(i)
                
            new_pairs = []
            for i in pair_index:
                new_pairs.append(pairs[i])
            
            monthsofexperience = []
            
            for i,j in new_pairs:
                monthsofexperience.append(get_number_of_months_from_dates(dates[i], dates[j]))
            
            monthsofexperience = np.absolute(monthsofexperience)
            experience_years = sum(monthsofexperience)/12
        
            return abs(experience_years)
        except:
            return 0
    
    def get_number_of_months_from_dates(date1, date2):
        if date2.lower() == 'present':
            date2 = datetime.now().strftime('%b %Y')
        try:
            if len(date1.split()[0]) > 3:
                date1 = date1.split()
                date1 = date1[0][:3] + ' ' + date1[1]
            if len(date2.split()[0]) > 3:
                date2 = date2.split()
                date2 = date2[0][:3] + ' ' + date2[1]
        except IndexError:
            return 0
        try:
            date1 = datetime.strptime(str(date1), '%b %Y')
            date2 = datetime.strptime(str(date2), '%b %Y')
            months_of_experience = relativedelta.relativedelta(date2, date1)
            months_of_experience = (months_of_experience.years * 12 + months_of_experience.months)
        except ValueError:
            return 0
      
        return months_of_experience
    
    SKILLS_DB = ['machine learning','data science','python', 'word','excel', 'deep learning', "data analytics", "time series", "financial analysis", "statistical modelling", 'python', 'r', 'mysql', 'powerbi', 'flask', 'html', 'azure ml Studio', 'microsoft office', 'visual studio', 'pyspark','tensorflow', "java", "c++", "javascript", "c"]

    def extract_skills(text):
        word_tokens = nltk.tokenize.word_tokenize(text)
        bigrams_trigrams = list(map(' '.join, nltk.everygrams( word_tokens, 2, 3)))

        # we create a set to keep the results in.
        found_skills = []

        # we search for each token in our skills database
        for token in  word_tokens:
            if token.lower() in SKILLS_DB:
                found_skills.append(token.lower())

        # we search for each bigram and trigram in our skills database
        for ngram in bigrams_trigrams:
            if ngram.lower() in SKILLS_DB:
                found_skills.append(ngram.lower())

        return found_skills

In [4]:
dir_list = os.listdir(r'D:\EYPROJECT2\Resumes')
res_list = []
paths = r'D:\EYPROJECT2\Resumes'
c = 0
for i in dir_list:
    c = c + 1
    if c == 5000:
        break
    pfinal = os.path.join(paths, i)
    data = ResumeParserTool(pfinal).get_extracted_data()
    res_list.append(data)

In [5]:
import pandas as pd

df = pd.DataFrame(res_list)

df

,name,email,mobile_number,skills,degree,designation,total_experience,area_of_study
0,None,None,None,[],[],[],0.000000,[]
1,Bay Area,professionalemail@resumeworded.com,None,"[python, data analytics]","[BE, BE]","[Analyst, Analyst, Analyst]",0.000000,[]
2,THOMAS JAMES,Thomas.James@university.edu,777.777.7777,"[excel, excel]",[BACHELOR OF SCIENCE],"[Analyst, Intern]",0.000000,[]
3,Stephen Greet,stephen@beamjobs.com,456-7890,"[python, python, excel]",[B.S],"[Analyst, Analyst, Analyst, Analyst]",3.916667,[MATHEMATICS]
4,John Regal,email@example.com,000-000-0000,[],[BACHELOR OF SCIENCE],"[Analyst, Analyst, Analyst, Analyst]",0.000000,[]
5,Marshville Road,info@qwikresume.com,None,"[excel, excel, excel]",[],"[Analyst, Analyst, Analyst, analyst]",0.000000,[]
6, Extract,info@qwikresumc.com,None,"[excel, microsoft office, data analytics]",[],"[Analyst, analyst]",0.000000,[IT]
7,ROBERT SMITH,info@qwikresume.com,None,[],[],"[Analyst, Analyst]",6.583333,[STATISTICS]
8,Stephen Greet,stephen@beamjobs.com,456-7890,"[python, excel, excel, mysql, python, excel]",[B.S],"[Analyst, Analyst, Intern]",2.583333,"[MATHEMATICS, STATISTICS]"
9,JANE FOSTER,someone@example.com,678-555-0103,"[python, java, machine learning]","[B.SC, M.S]",[],1.000000,"[IT, IT, IT]"


In [6]:
bins = [-1, 2, 4, 7, np.inf]
names = ['Analyst', 'associate consultant', 'Consultant', 'Senior Consultant']

df['Experience_Range'] = pd.cut(df['total_experience'], bins, labels=names)
df['Status'] = df.apply(lambda _: '', axis=1)
df['Review Status'] = df.apply(lambda _: '', axis=1)
df['Reason for rejection'] = df.apply(lambda _: '', axis=1)
df['Manager'] = df.apply(lambda _: 'N/A', axis=1)
df['Interviewer'] = df.apply(lambda _: '10 minutes', axis=1)
df['Technical Interviewer'] = df.apply(lambda _: '30 minutes', axis=1)
df['Source'] = df.apply(lambda _: '', axis=1)
df['Review Comments'] = df.apply(lambda _: '', axis=1)

In [7]:
df

,name,email,mobile_number,skills,degree,designation,total_experience,area_of_study,Experience_Range,Status,Review Status,Reason for rejection,Manager,Interviewer,Technical Interviewer,Source,Review Comments
0,None,None,None,[],[],[],0.000000,[],Analyst,,,,N/A,10 minutes,30 minutes,,
1,Bay Area,professionalemail@resumeworded.com,None,"[python, data analytics]","[BE, BE]","[Analyst, Analyst, Analyst]",0.000000,[],Analyst,,,,N/A,10 minutes,30 minutes,,
2,THOMAS JAMES,Thomas.James@university.edu,777.777.7777,"[excel, excel]",[BACHELOR OF SCIENCE],"[Analyst, Intern]",0.000000,[],Analyst,,,,N/A,10 minutes,30 minutes,,
3,Stephen Greet,stephen@beamjobs.com,456-7890,"[python, python, excel]",[B.S],"[Analyst, Analyst, Analyst, Analyst]",3.916667,[MATHEMATICS],associate consultant,,,,N/A,10 minutes,30 minutes,,
4,John Regal,email@example.com,000-000-0000,[],[BACHELOR OF SCIENCE],"[Analyst, Analyst, Analyst, Analyst]",0.000000,[],Analyst,,,,N/A,10 minutes,30 minutes,,
5,Marshville Road,info@qwikresume.com,None,"[excel, excel, excel]",[],"[Analyst, Analyst, Analyst, analyst]",0.000000,[],Analyst,,,,N/A,10 minutes,30 minutes,,
6, Extract,info@qwikresumc.com,None,"[excel, microsoft office, data analytics]",[],"[Analyst, analyst]",0.000000,[IT],Analyst,,,,N/A,10 minutes,30 minutes,,
7,ROBERT SMITH,info@qwikresume.com,None,[],[],"[Analyst, Analyst]",6.583333,[STATISTICS],Consultant,,,,N/A,10 minutes,30 minutes,,
8,Stephen Greet,stephen@beamjobs.com,456-7890,"[python, excel, excel, mysql, python, excel]",[B.S],"[Analyst, Analyst, Intern]",2.583333,"[MATHEMATICS, STATISTICS]",associate consultant,,,,N/A,10 minutes,30 minutes,,
9,JANE FOSTER,someone@example.com,678-555-0103,"[python, java, machine learning]","[B.SC, M.S]",[],1.000000,"[IT, IT, IT]",Analyst,,,,N/A,10 minutes,30 minutes,,


In [8]:
df.to_csv("Resumes.csv")